In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path


BASE_DIR = Path().resolve().parent
DATA_PATH = BASE_DIR / "data/raw/ml-100k"
COLUMNS = [
    "movie_id",
    "title",
    "release_date",
    "video_release_date",
    "IMDb_URL",
    "unknown",
    "Action",
    "Adventure",
    "Animation",
    "Children's",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "Film-Noir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "Sci-Fi",
    "Thriller",
    "War",
    "Western"
]

df = pd.read_csv(
    DATA_PATH / "u.item",
    sep="|",
    header=None,
    encoding="latin-1",
    names=COLUMNS,
)
df.head()

,movie_id,title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [2]:
def build_df(df: pd.DataFrame) -> pd.DataFrame:
    titles = df["title"].tolist()
    descriptions = []
    for _, row in df.iterrows():
        genres = []
        for genre in COLUMNS[5:]:
            if row[genre] == 1:
                genres.append(genre)
        genres = " ".join(genres)
        descriptions.append(genres)

    res_df = pd.DataFrame(
        {"title": titles, "description": descriptions},
    )
    return res_df


movies_data = build_df(df)
movies_data.head()

,title,description
0,Toy Story (1995),Animation Children's Comedy
1,GoldenEye (1995),Action Adventure Thriller
2,Four Rooms (1995),Thriller
3,Get Shorty (1995),Action Comedy Drama
4,Copycat (1995),Crime Drama Thriller


In [3]:
# Векторизуем описания фильмов с помощью TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(movies_data["description"])

# 3. Считаем матрицу косинусного сходства между всеми фильмами
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [6]:
# 4. Функция для поиска рекомендаций
def get_recommendations(movie_title, cosine_sim_matrix, dataframe, top_n=5):
    # Находим индекс фильма по его названию
    movie_idx = dataframe[dataframe["title"] == movie_title].index[0]

    # Получаем строку сходства этого фильма со всеми остальными
    sim_scores = list(enumerate(cosine_sim_matrix[movie_idx]))

    # Сортируем фильмы по убыванию сходства (первым будет сам этот фильм, у него сходство 1.0)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Берем топ_н фильмов (исключая самый первый — сам себя)
    recommended_indices = [i[0] for i in sim_scores[1 : top_n + 1]]

    # Возвращаем названия рекомендованных фильмов
    return dataframe["title"].iloc[recommended_indices].tolist()


# --- Тестируем систему ---
target_movie = "GoldenEye (1995)"
recommendations = get_recommendations(target_movie, cosine_sim, df)

print(f"Поскольку вы смотрели '{target_movie}', вам могут понравиться:")
for movie in recommendations:
    print(f"- {movie}")

Поскольку вы смотрели 'GoldenEye (1995)', вам могут понравиться:
- Rock, The (1996)
- Twister (1996)
- Clear and Present Danger (1994)
- Daylight (1996)
- Chain Reaction (1996)
